# 43 Series Solutions and Linear Systems

| | |
|---|---|
| **Course** | Differential Equations |
| **Prerequisite** | 25 and 39 |

About an ordinary point, look for a solution that is a power series

$$
y=\sum_{n=0}^{\infty} a_n x^n.
$$

Differentiate term by term, substitute into the equation, and equate coefficients of like powers of $x$.
The coefficients $a_0$ and $a_1$ usually remain free. They are the two constants fixed by the initial conditions.

A first-order linear system $\mathbf{x}'=A\mathbf{x}$ is solved by eigenvalues.
If $\lambda$ is an eigenvalue and $\mathbf{v}$ is a corresponding eigenvector, then $e^{\lambda t}\mathbf{v}$ is a solution.


## Example 1 — A series that recovers the exponential

### Hand solution

Solve $y'=y$ with $y(0)=1$.
Write $y=\sum_{n=0}^{\infty} a_n x^n$ with $a_0=1$. Term-by-term differentiation and matching of powers give $n a_n=a_{n-1}$, so $a_n=a_{n-1}/n$.

$$
\begin{aligned}
\sum_{n=1}^{\infty} n a_n x^{n-1}
  &= \sum_{n=0}^{\infty} a_n x^n, \\
n a_n &= a_{n-1}, \\
a_n &= \dfrac{1}{n!}, \\
y &= \sum_{n=0}^{\infty}\dfrac{x^n}{n!}=e^{x}.
\end{aligned}
$$

### Python recheck


In [1]:
# Load SymPy for the symbolic check.
import sympy as sp
# Independent variable and unknown function.
x = sp.symbols("x")
y = sp.Function("y")
# Equation y' = y.
ode = sp.Eq(y(x).diff(x), y(x))
# Initial condition y(0) = 1.
sol = sp.dsolve(ode, y(x), ics={y(0): 1})
# Taylor polynomial of exp(x) through degree 5.
series = sp.series(sp.exp(x), x, 0, 6).removeO()
print(sol)
print(series)
# The closed form is exp(x).
assert sp.simplify(sol.rhs - sp.exp(x)) == 0
# The first six Taylor terms match 1 + x + ... + x^5/120.
assert series == 1 + x + x**2/2 + x**3/6 + x**4/24 + x**5/120
print("Confirmed")


Eq(y(x), exp(x))
x**5/120 + x**4/24 + x**3/6 + x**2/2 + x + 1
Confirmed


## Example 2 — The recurrence for $y''-xy=0$, a start

### Hand solution

Airy's equation $y''-xy=0$ has an ordinary point at $0$.
Substitute $y=\sum_{n=0}^{\infty} a_n x^n$:

$$
\sum_{n=2}^{\infty} n(n-1)a_n x^{n-2}
-\sum_{n=0}^{\infty} a_n x^{n+1}
=0.
$$

The coefficient of $x^0$ forces $a_2=0$. For every higher power, the coefficient is determined by the coefficient three steps earlier.
The two free coefficients $a_0$ and $a_1$ produce two independent solutions.
The first terms of the solution with $a_0=1$ and $a_1=0$ are

$$
y=1+\dfrac{1}{6}x^3+\cdots.
$$

The equation itself gives $y''(0)=0$. Differentiating $y''=xy$ produces $y'''=y+xy'$, and the value at zero is $y'''(0)=1$.
The cubic term is therefore the Taylor term $\dfrac{y'''(0)}{6}x^3$.

### Python recheck


In [2]:
# Load SymPy for the symbolic check.
import sympy as sp
# Independent variable and unknown function.
x = sp.symbols("x")
y = sp.Function("y")
# Airy's equation y'' - x y = 0.
ode = sp.Eq(y(x).diff(x, 2) - x*y(x), 0)
# Power series about the ordinary point 0, through order 5.
series = sp.dsolve(ode, y(x), hint="2nd_power_series_ordinary", n=6)
# Drop the big-O term and read off the free constants.
polynomial = series.rhs.removeO()
constants = sorted(polynomial.free_symbols - {x}, key=str)
print(series)
# The solution with a0 = 1 and a1 = 0.
particular = polynomial.subs({constants[0]: 0, constants[1]: 1})
# Check y(0) = 1, the first derivative at 0, and the third derivative at 0.
assert particular.subs(x, 0) == 1
# The first derivative at 0 vanishes for this choice of constants.
assert sp.diff(particular, x).subs(x, 0) == 0
# The third derivative at 0 equals 1, which produces the x^3/6 term.
assert sp.diff(particular, x, 3).subs(x, 0) == 1
print(particular)
print("Confirmed")


Eq(y(x), C2*(x**3/6 + 1) + C1*x*(x**3/12 + 1) + O(x**6))
x**3/6 + 1
Confirmed


## Example 3 — A system and an eigenvalue

### Hand solution

Solve the system

$$
\begin{aligned}
x' &= y, \\
y' &= -x,
\end{aligned}
$$

with $x(0)=1$ and $y(0)=0$.
Differentiate the first equation and substitute the second:

$$
\begin{aligned}
x'' &= y' = -x, \\
x''+x &= 0, \\
x &= \cos t, \\
y &= x' = -\sin t.
\end{aligned}
$$

The coefficient matrix $\begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}$ has eigenvalues $\pm i$, and the real solutions built from those eigenvalues are the same sine and cosine.

### Python recheck


In [3]:
# Load SymPy for the symbolic check.
import sympy as sp
# Independent variable and the two unknown functions.
t = sp.symbols("t")
x = sp.Function("x")
y = sp.Function("y")
# First-order system x' = y, y' = -x.
system = [
    sp.Eq(x(t).diff(t), y(t)),
    sp.Eq(y(t).diff(t), -x(t)),
]
# Initial conditions x(0) = 1, y(0) = 0.
sol = sp.dsolve(system, [x(t), y(t)], ics={x(0): 1, y(0): 0})
print(sol)
# Coefficient matrix and its eigenvalues.
matrix = sp.Matrix([[0, 1], [-1, 0]])
eigenvalues = matrix.eigenvals()
print(eigenvalues)
# The solution is x = cos(t), y = -sin(t).
assert sp.simplify(sol[0].rhs - sp.cos(t)) == 0
# The second unknown is minus sin(t).
assert sp.simplify(sol[1].rhs + sp.sin(t)) == 0
print("Confirmed")


[Eq(x(t), cos(t)), Eq(y(t), -sin(t))]
{-I: 1, I: 1}
Confirmed


## Example 4 — Two real eigenvalues

### Hand solution

Solve

$$
\mathbf{u}'
=\begin{pmatrix} 1 & 0 \\ 0 & -2 \end{pmatrix}
\mathbf{u}.
$$

The eigenvalues are $1$ and $-2$, with eigenvectors $(1,0)$ and $(0,1)$. The general solution is

$$
\mathbf{u}
=c_1 e^{t}\begin{pmatrix} 1 \\ 0 \end{pmatrix}
+c_2 e^{-2t}\begin{pmatrix} 0 \\ 1 \end{pmatrix}.
$$

The initial condition $\mathbf{u}(0)=(3,4)$ forces $c_1=3$ and $c_2=4$.

### Python recheck


In [4]:
# Load SymPy for the symbolic check.
import sympy as sp
# Independent variable.
t = sp.symbols("t")
# Diagonal coefficient matrix.
A = sp.Matrix([[1, 0], [0, -2]])
# The first reported eigenvalue is one of the two real eigenvalues.
assert A.eigenvects()[0][0] in (1, -2)
# Solution for the initial state (3, 4).
u = sp.Matrix([3*sp.exp(t), 4*sp.exp(-2*t)])
# The vector u satisfies u' = A u and the initial condition.
assert sp.simplify(u.diff(t) - A*u) == sp.Matrix([0, 0])
# The same vector meets the initial condition (3, 4).
assert u.subs(t, 0) == sp.Matrix([3, 4])
print(u)
print("Confirmed")


Matrix([[3*exp(t)], [4*exp(-2*t)]])
Confirmed


## A pitfall

Extend the series at least far enough for the recurrence to show itself.
A list of six coefficients, with no relation among them, is only a polynomial snapshot of the solution.
In a linear system, each eigenvalue determines one mode. The initial condition fixes the coefficient of each mode.

## Summary

When the coefficients of the equation are analytic, a power series about an ordinary point produces two independent solutions.
A linear system with a constant matrix reduces to an eigenvalue problem and its eigenvectors.
